In [ ]:
#| default_exp deploy

# Using the hardware

> Kokoro's decoder on the Mac's GPU and Neural Engine through Core ML, with smaller weights, and on XNNPACK for Android.

Everything so far ran Kokoro the way it ships: PyTorch on the CPU, 32-bit weights, 327 MB. Phones have more than a CPU:
a GPU, and a neural accelerator (Apple's Neural Engine, or an NPU on Android) built for exactly the convolutions a
vocoder is made of. This notebook finds out what it takes to use them, measuring speed and quality at every step so
that nothing gets faster by quietly getting worse.

It's marked `skip_exec`, since it converts models and scores a minute of audio many times over: about twenty minutes.

In [ ]:
#| export
import copy
import types

import coremltools as ct
import numpy as np
import torch
import torch.nn.functional as F
from coremltools.converters.mil import Builder as mb
from coremltools.converters.mil.frontend.torch.ops import _get_inputs, mul
from coremltools.converters.mil.frontend.torch.torch_op_registry import register_torch_op
from coremltools.converters.mil.mil import types as mil_types
from kokoro.custom_stft import CustomSTFT
from kokoro.istftnet import TorchSTFT

from text_to_speech.pretrained import SR, pipeline

scikit-learn version 1.9.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
Torch version 2.14.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.7.0 is the most recent version that has been tested.
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torchao/quantization/quant_api.py:1558: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.
W1101 21:23:24.113000 74736 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W1101 21:23:24.152000 74736 .venv/lib/python3.13/site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is depr

In [ ]:
import os
import tempfile
import time
from collections import Counter

import coremltools.optimize.coreml as cto
import librosa
import pandas as pd
from coremltools.models.compute_plan import MLComputePlan

from text_to_speech.data import metadata
from text_to_speech.pretrained import say

## Where the time goes

Kokoro has five stages. The first four turn phonemes into 25 ms frames of features, pitch (F0) and energy (N). The
last, the decoder, is a vocoder: it turns those frames into audio. `decoder_inputs` runs the first four, which is
Kokoro's `forward_with_tokens` up to the decoder.

In [ ]:
#| export
model = pipeline.model


@torch.no_grad()
def decoder_inputs(phonemes, voice="af_heart"):
    "Everything Kokoro computes before the decoder: frame features, pitch, energy, and the voice's style vector."
    ref_s = pipeline.load_voice(voice)[len(phonemes) - 1]
    ids = torch.LongTensor([[0, *[model.vocab[p] for p in phonemes if p in model.vocab], 0]])
    n = ids.shape[1]
    lengths, mask = torch.tensor([n]), torch.zeros(1, n, dtype=torch.bool)
    s = ref_s[:, 128:]
    bert = model.bert_encoder(model.bert(ids, attention_mask=(~mask).int())).transpose(-1, -2)
    d = model.predictor.text_encoder(bert, s, lengths, mask)
    duration = torch.sigmoid(model.predictor.duration_proj(model.predictor.lstm(d)[0])).sum(-1).squeeze()
    frames = torch.round(duration).clamp(min=1).long()
    aln = torch.zeros(1, n, int(frames.sum()))
    aln[0, torch.repeat_interleave(torch.arange(n), frames), torch.arange(int(frames.sum()))] = 1
    F0, N = model.predictor.F0Ntrain(d.transpose(-1, -2) @ aln, s)
    return model.text_encoder(ids, lengths, mask) @ aln, F0, N, ref_s[:, :128].contiguous()

In [ ]:
text = "Long paragraphs are split so pause and seek stay responsive, and the reader follows along word by word."
phonemes = say(text).phonemes
t = time.perf_counter()
for _ in range(5):
    x = decoder_inputs(phonemes)
front = (time.perf_counter() - t) / 5
with torch.no_grad():
    t = time.perf_counter()
    for _ in range(5):
        audio = model.decoder(*x)
decode = (time.perf_counter() - t) / 5
print(f"{audio.numel() / SR:.1f}s of audio: stages 1-4 {front * 1000:.0f} ms, decoder {decode * 1000:.0f} ms")
for name in ["bert", "bert_encoder", "predictor", "text_encoder", "decoder"]:
    print(f"{name:<13} {sum(p.numel() for p in getattr(model, name).parameters()) / 1e6:5.1f}M parameters")

7.0s of audio: stages 1-4 76 ms, decoder 760 ms
bert            6.3M parameters
bert_encoder    0.4M parameters
predictor      16.2M parameters
text_encoder    5.6M parameters
decoder        53.3M parameters


The decoder is about 90% of the time and two thirds of the weights. So optimising Kokoro means optimising the decoder,
and the front half can stay on the CPU anywhere. The decoder is almost all one-dimensional convolutions, exactly what
GPUs and neural accelerators are for.

## The bar

The decoder makes noise on purpose: its pitch-driven source has a random starting phase and added noise. Two runs at
32-bit with different seeds already differ, so that's the bar. Anything we do should keep its output about as close to
the original as two seeds are to each other, measured as the mean distance between log mel spectrograms, and keep
UTMOS's predicted MOS. The test set is the ten sentences from notebooks 100 and 101.

In [ ]:
test_sentence = "Speed comes from the voice itself, not from stretching the audio afterwards."
texts = [
    test_sentence,
    *[t for t in metadata().query("id.str.startswith('LJ050')").normalized_text if len(t) < 200][:9],
]
inputs = [decoder_inputs(say(t).phonemes) for t in texts]
utmos = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).eval()


def mel(a):
    return np.log(np.maximum(librosa.feature.melspectrogram(y=a, sr=SR, n_fft=1024, hop_length=256, n_mels=80), 1e-5))


def distance(a, b):
    n = min(len(a), len(b))
    return np.abs(mel(a[:n]) - mel(b[:n])).mean()


@torch.no_grad()
def mos(a):
    return utmos(torch.tensor(a, dtype=torch.float32)[None], SR).item()


@torch.no_grad()
def original(seed):
    torch.manual_seed(seed)
    return [model.decoder(*x).squeeze().numpy() for x in inputs]


ref, other_seed = original(0), original(1)
seconds = sum(map(len, ref)) / SR
floor = np.mean([distance(a, b) for a, b in zip(ref, other_seed)])
print(f"{seconds:.1f}s of audio; two seeds differ by {floor:.3f}; MOS {np.mean([mos(a) for a in ref]):.2f}")
results = []


@torch.no_grad()
def evaluate(name, decode, repeats=2):
    "decode(asr, F0, N, s) -> audio. Times it on all ten sentences and scores it against the original."
    decode(*inputs[0])
    t = time.perf_counter()
    for _ in range(repeats):
        outs = [np.asarray(decode(*x), dtype=np.float32).squeeze() for x in inputs]
    elapsed = (time.perf_counter() - t) / repeats
    d = np.mean([distance(a, b) for a, b in zip(outs, ref)])
    results.append(
        {
            "decoder": name,
            "x real time": seconds / elapsed,
            "vs seed floor": float(d / floor),
            "MOS": float(np.mean([mos(a) for a in outs])),
        }
    )
    print(" | ".join(f"{v:.2f}" if isinstance(v, float) else v for v in results[-1].values()))


def compiled(m, units="CPU_ONLY"):
    "Save, compile and load a converted model. Predicting with ct.convert's in-memory model crashed later work."
    path = os.path.join(tempfile.mkdtemp(), "model.mlpackage")
    m.save(path)
    return ct.models.CompiledMLModel(ct.utils.compile_model(path), compute_units=getattr(ct.ComputeUnit, units))

61.8s of audio; two seeds differ by 0.070; MOS 4.52


Using cache found in /Users/henrydashwood/.cache/torch/hub/tarepan_SpeechMOS_v1.2.0
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


## Cheap wins on the CPU

Threads first. This M3 Pro has 6 performance cores, which is PyTorch's default.

In [ ]:
for n in [1, 6]:
    torch.set_num_threads(n)
    evaluate(f"PyTorch CPU, {n} thread{'s' if n > 1 else ''}", lambda *x: model.decoder(*x))

PyTorch CPU, 1 thread | 5.57 | 0.99 | 4.52
PyTorch CPU, 6 threads | 9.45 | 1.00 | 4.52


One core already manages about 5x real time, and six only get to about 9x: the decoder doesn't parallelise well on the
CPU. 16-bit doesn't help either. PyTorch has no fast 16-bit convolutions on the CPU, and `torch.autocast` to bfloat16
ran at 0.1x real time (not rerun here: it takes ten minutes). `torch.compile` was slower than eager.

## The STFT

The decoder doesn't produce audio directly. Its last layer predicts a tiny spectrogram, 11 frequency bins every 5
samples, and an inverse STFT turns that into a waveform. It also takes an STFT of its pitch source on the way in.
Kokoro does both with `torch.stft`, which uses complex numbers. Exporters don't do complex numbers, and neither does
the Mac's GPU, quickly.

Kokoro ships `CustomSTFT` for exactly this, built from convolutions. But its inverse is an approximation (its comments
say so), and it pads differently. Compared with `torch.stft` on a second of noise:

In [ ]:
x = torch.randn(1, SR)
original_stft, custom = TorchSTFT(20, 5, 20), CustomSTFT(20, 5, 20)
m0, p0 = original_stft.transform(x)
y0 = original_stft.inverse(m0, p0).squeeze()
m, p = custom.transform(x)
print(f"forward max error {((m * torch.exp(1j * p)) - (m0 * torch.exp(1j * p0))).abs().max():.2e}")
print(f"inverse max error {(custom.inverse(m0, p0).squeeze() - y0).abs().max():.2e} on a signal of std {x.std():.2f}")

forward max error 1.64e+00
inverse max error 1.02e+00 on a signal of std 0.99


Two fixes. The forward transform is right once it pads by reflection, as `torch.stft` does. The inverse needs two
things it skips: every frequency bin except the first and last has a mirror image, so counts twice, and the
overlapping windows don't add up to a constant, so the result must be divided by their sum. That's what
`torch.istft` does, written as convolutions:

In [ ]:
#| export
class ExactSTFT(CustomSTFT):
    "Kokoro's convolutional STFT with torch.stft's reflect padding, and an inverse that matches torch.istft."

    def __init__(self, n_fft, hop):
        super().__init__(n_fft, hop, n_fft, pad_mode="reflect")
        k = np.arange(n_fft // 2 + 1)
        scale = np.where((k == 0) | (k == n_fft // 2), 1, 2) / n_fft  # Bins with a mirror image count twice
        angle = 2 * np.pi * np.outer(k, np.arange(n_fft)) / n_fft
        window = self.window.numpy()
        self.weight_backward_real = torch.tensor(np.cos(angle) * scale[:, None] * window, dtype=torch.float32)[:, None]
        self.weight_backward_imag = torch.tensor(np.sin(angle) * scale[:, None] * window, dtype=torch.float32)[:, None]
        self.register_buffer("window_squared", (self.window**2)[None, None])

    def inverse(self, magnitude, phase):
        real = F.conv_transpose1d(magnitude * torch.cos(phase), self.weight_backward_real, stride=self.hop_length)
        imag = F.conv_transpose1d(magnitude * torch.sin(phase), self.weight_backward_imag, stride=self.hop_length)
        # Overlapping windows add up unevenly: divide by their total, as torch.istft does
        envelope = F.conv_transpose1d(torch.ones_like(magnitude[:, :1]), self.window_squared, stride=self.hop_length)
        pad = self.n_fft // 2
        return ((real - imag) / envelope)[..., pad:-pad]

In [ ]:
exact = ExactSTFT(20, 5)
m, p = exact.transform(x)
print(f"forward max error {((m * torch.exp(1j * p)) - (m0 * torch.exp(1j * p0))).abs().max():.2e}")
print(f"inverse max error {(exact.inverse(m0, p0).squeeze() - y0).abs().max():.2e}")

forward max error 2.34e-06
inverse max error 7.15e-07


## Two more things exporters trip on

The pitch source turns F0 into phase by adding it up over the whole sentence, then takes the sine. Two details matter
once it leaves PyTorch:

- Kokoro wraps the phase increment with `% 1`. Core ML's converter treats `% 1` as integer arithmetic and returns 0,
  and in silence the predicted F0 can be a hair below zero, where `%` behaves differently anyway. Clamping F0 at zero
  and writing `x - floor(x)` means the same and survives conversion.
- The accumulated phase reaches thousands of cycles. That's fine in 32-bit floats, and hopeless in 16-bit, which
  resolves numbers in the thousands only to the nearest few. So that part has to stay 32-bit.

Here's the `%` problem on its own:

In [ ]:
class Wrap(torch.nn.Module):
    def forward(self, x):
        return x % 1


x = torch.tensor([[0.25, 1.5, 0.001]])
converted = ct.convert(
    torch.jit.trace(Wrap(), (x,)),
    inputs=[ct.TensorType(name="x", shape=x.shape)],
    minimum_deployment_target=ct.target.macOS15,
    compute_precision=ct.precision.FLOAT32,
    compute_units=ct.ComputeUnit.CPU_ONLY,
)
print(
    "PyTorch:",
    Wrap()(x).tolist(),
    " Core ML:",
    next(iter(compiled(converted).predict({"x": x.numpy()}).values())).tolist(),
)

PyTorch: [[0.25, 0.5, 0.0010000000474974513]]  Core ML: [[0.0, 0.0, 0.0]]


/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Model is not in eval mode. Consider calling '.eval()' on your model prior to conversion
Running MIL default pipeline:   0%|          | 0/92 [00:00<?, ? passes/s]/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '3', of the source model, has been renamed to 'var_3' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
Running MIL backend_mlprogram pipeline: 100%|##########| 12/12 [00:00<00:00, 61832.49 passes/s]


So `exportable_decoder` makes a copy of the decoder with the exact STFT, a sine source that converts, and weight
normalisation folded away. Kokoro computes every convolution's weights from two tensors on each call; folding does
that once, so exporters see constant weights.

In [ ]:
#| export
def _f02sine(self, f0_values):
    "Kokoro's SineGen._f02sine, with negative F0 clamped, % written out, and the phase wrapped before the sine."
    rad_values = f0_values.clamp(min=0) / self.sampling_rate  # F0 can be a hair below 0 in silence
    rad_values = rad_values - torch.floor(rad_values)  # % 1, which Core ML gets wrong
    rand_ini = torch.rand(f0_values.shape[0], f0_values.shape[2], device=f0_values.device)
    rand_ini[:, 0] = 0
    rad_values[:, 0, :] = rad_values[:, 0, :] + rand_ini
    rad_values = F.interpolate(rad_values.transpose(1, 2), scale_factor=1 / self.upsample_scale, mode="linear")
    cycles = torch.cumsum(rad_values.transpose(1, 2), dim=1)
    cycles = F.interpolate(
        cycles.transpose(1, 2) * self.upsample_scale, scale_factor=self.upsample_scale, mode="linear"
    )
    cycles = cycles.transpose(1, 2)
    return torch.sin(2 * torch.pi * (cycles - torch.floor(cycles)))


def exportable_decoder():
    "A copy of Kokoro's decoder that exporters can handle, and that sounds the same."
    for m in model.decoder.modules():
        if hasattr(
            m, "weight_g"
        ):  # weight_norm caches a computed weight, which deepcopy refuses. It's recomputed every call
            m.weight = m.weight.detach()
    dec = copy.deepcopy(model.decoder).eval().requires_grad_(False)
    dec.generator.stft = ExactSTFT(dec.generator.stft.filter_length, dec.generator.stft.hop_length)
    sinegen = dec.generator.m_source.l_sin_gen
    sinegen._f02sine = types.MethodType(_f02sine, sinegen)
    for m in dec.modules():
        if hasattr(m, "weight_g"):
            torch.nn.utils.remove_weight_norm(m)
    return dec

In [ ]:
dec = exportable_decoder()
evaluate("PyTorch CPU, exportable", lambda *x: dec(*x))
gpu = copy.deepcopy(dec).to("mps")
evaluate("PyTorch GPU, exportable", lambda *x: gpu(*[t.to("mps") for t in x]).cpu())

PyTorch CPU, exportable | 10.04 | 2.35 | 4.52
PyTorch GPU, exportable | 25.84 | 2.36 | 4.52


Same MOS, and the distance is a little above the floor, from tiny phase differences where the source is silent. And
without complex numbers the Mac's GPU is now well ahead of the CPU, where notebook 100 found it no faster.

## Split in two

The 32-bit-only part, the pitch source and its STFT, is a tiny fraction of the work. So the decoder becomes two
models: `HarmonicSource`, F0 to the source's spectrum, which stays 32-bit on the CPU; and `Body`, everything else,
which can be 16-bit on whatever hardware is fastest. `Body` is Kokoro's own `Decoder.forward` and `Generator.forward`
with the source as an input.

In [ ]:
#| export
class HarmonicSource(torch.nn.Module):
    "F0 -> the STFT of the decoder's pitch-driven source. Adds up phase over the whole sentence, so needs 32-bit."

    def __init__(self, dec):
        super().__init__()
        self.gen = dec.generator

    def forward(self, F0_curve):
        f0 = self.gen.f0_upsamp(F0_curve[:, None]).transpose(1, 2)
        har_source, _, _ = self.gen.m_source(f0)
        spec, phase = self.gen.stft.transform(har_source.transpose(1, 2).squeeze(1))
        return torch.cat([spec, phase], dim=1)


class Body(torch.nn.Module):
    "The rest of the decoder: Kokoro's Decoder.forward and Generator.forward, taking the source as an input."

    def __init__(self, dec):
        super().__init__()
        self.dec = dec

    def forward(self, asr, F0_curve, N, s, har):
        d, g = self.dec, self.dec.generator
        F0, N = d.F0_conv(F0_curve.unsqueeze(1)), d.N_conv(N.unsqueeze(1))
        x = d.encode(torch.cat([asr, F0, N], axis=1), s)
        asr_res, res = d.asr_res(asr), True
        for block in d.decode:
            if res:
                x = torch.cat([x, asr_res, F0, N], axis=1)
            x = block(x, s)
            res = res and block.upsample_type == "none"
        for i in range(g.num_upsamples):
            x = F.leaky_relu(x, negative_slope=0.1)
            x_source = g.noise_res[i](g.noise_convs[i](har), s)
            x = g.ups[i](x)
            if i == g.num_upsamples - 1:
                x = g.reflection_pad(x)
            x = x + x_source
            x = sum(g.resblocks[i * g.num_kernels + j](x, s) for j in range(g.num_kernels)) / g.num_kernels
        x = g.conv_post(F.leaky_relu(x))
        spec, phase = torch.exp(x[:, : g.post_n_fft // 2 + 1, :]), torch.sin(x[:, g.post_n_fft // 2 + 1 :, :])
        return g.stft.inverse(spec, phase)

## Core ML

Core ML wants fixed shapes, so we convert each model at three sizes, 128, 256 and 512 frames (3.2, 6.4 and 12.8 s),
and pad each sentence up to the next. Two more converter gaps need filling: Kokoro takes `rsqrt` of the integer 2,
which PyTorch quietly makes a float and coremltools refuses, and it spells multiplication `torch.multiply`.

In [ ]:
#| export
@register_torch_op(override=True)
def rsqrt(context, node):
    "Kokoro takes rsqrt of the integer 2, which PyTorch promotes to a float and coremltools doesn't."
    x = _get_inputs(context, node, expected=1)[0]
    if mil_types.is_int(x.dtype):
        x = mb.cast(x=x, dtype="fp32")
    context.add(mb.rsqrt(x=x, name=node.name))


@register_torch_op
def multiply(context, node):
    "torch.multiply is another name for `*`."
    mul(context, node)


BUCKETS = [128, 256, 512]  # Frames of 25 ms: 3.2, 6.4 and 12.8 s


def to_coreml(module, args, names, precision=ct.precision.FLOAT16):
    "Trace `module` at the shapes of `args` and convert it."
    traced = torch.jit.trace(module, args, check_trace=False)
    return ct.convert(
        traced,
        inputs=[ct.TensorType(name=n, shape=a.shape) for n, a in zip(names, args)],
        outputs=[ct.TensorType(name="out")],
        compute_precision=precision,
        minimum_deployment_target=ct.target.macOS15,
    )


def example(T):
    "Inputs for a T-frame bucket."
    asr, F0, N, s = torch.zeros(1, 512, T), torch.zeros(1, 2 * T), torch.zeros(1, 2 * T), torch.zeros(1, 128)
    return asr, F0, N, s

In [ ]:
source, body = HarmonicSource(dec), Body(dec)
with torch.no_grad():
    sources = {T: to_coreml(source, (example(T)[1],), ["F0"], ct.precision.FLOAT32) for T in BUCKETS}
    bodies = {T: to_coreml(body, (*example(T), source(example(T)[1])), ["asr", "F0", "N", "s", "har"]) for T in BUCKETS}


def load(m, units):
    return compiled(m, units)


def coreml(bodies, units):
    "A decode function: the source on the CPU in 32-bit, the body on `units`."
    src = {T: load(m, "CPU_ONLY") for T, m in sources.items()}
    bod = {T: load(m, units) for T, m in bodies.items()}

    def decode(asr, F0, N, s):
        T = asr.shape[-1]
        B = next(b for b in BUCKETS if b >= T)
        pad = lambda x, n: F.pad(x, (0, n - x.shape[-1])).numpy()
        f0 = pad(F0, 2 * B)
        har = src[B].predict({"F0": f0})["out"]
        out = bod[B].predict({"asr": pad(asr, B), "F0": f0, "N": pad(N, 2 * B), "s": s.numpy(), "har": har})["out"]
        return out.squeeze()[: T * 600]

    return decode


for units in ["CPU_ONLY", "CPU_AND_GPU"]:
    evaluate(f"Core ML 16-bit, {units}", coreml(bodies, units))

Core ML 16-bit, CPU_ONLY | 25.99 | 7.88 | 4.48
Core ML 16-bit, CPU_AND_GPU | 36.70 | 6.70 | 4.52


/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Model is not in eval mode. Consider calling '.eval()' on your model prior to conversion
Running MIL backend_mlprogram pipeline: 100%|##########| 12/12 [00:00<00:00, 1169.82 passes/s]
Model is not in eval mode. Consider calling '.eval()' on your model prior to conversion
Running MIL backend_mlprogram pipeline: 100%|##########| 12/12 [00:00<00:00, 1224.26 passes/s]
Model is not in eval mode. Consider calling '.eval()' on your model prior to conversion
Running MIL backend_mlprogram pipeline: 100%|##########| 12/12 [00:00<00:00, 1193.94 passes/s]
/Users/henrydash

The GPU through Core ML is the fastest yet, at full quality. The Neural Engine is missing because it won't take
models this long: asking for it fails to build at these sizes, and `ALL`, which lets Core ML choose, crashed the
process trying.

## The Neural Engine

It has limits on tensor sizes, and the decoder's last layers run on 120 time steps for every frame. Asking Core ML's
compute plan where each operation would run, at smaller sizes:

In [ ]:
def plan(T):
    "Speed and where each op runs, for a body of T frames, on the CPU, GPU and Neural Engine."
    asr, F0, N, s = inputs[0]
    args = (asr[..., :T], F0[:, : 2 * T], N[:, : 2 * T], s)
    with torch.no_grad():
        args = (*args, source(args[1]))
        path = os.path.join(tempfile.mkdtemp(), "body.mlpackage")
        to_coreml(body, args, ["asr", "F0", "N", "s", "har"]).save(path)
    compiled = ct.utils.compile_model(path)
    feed = dict(zip(["asr", "F0", "N", "s", "har"], [a.numpy() for a in args]))
    for units in ["CPU_ONLY", "CPU_AND_GPU", "CPU_AND_NE"]:
        cp = MLComputePlan.load_from_path(compiled, compute_units=getattr(ct.ComputeUnit, units))
        ops = cp.model_structure.program.functions["main"].block.operations
        devices = Counter(
            type(cp.get_compute_device_usage_for_mlprogram_operation(op).preferred_compute_device).__name__[2:-13]
            for op in ops
            if cp.get_compute_device_usage_for_mlprogram_operation(op)
        )
        m = ct.models.CompiledMLModel(compiled, compute_units=getattr(ct.ComputeUnit, units))
        m.predict(feed)
        t = time.perf_counter()
        for _ in range(10):
            m.predict(feed)
        elapsed = (time.perf_counter() - t) / 10
        print(f"{T:>4} frames, {units:<12} {T * 0.025 / elapsed:5.1f}x real time  {dict(devices)}")


for T in [64, 128]:
    plan(T)

  64 frames, CPU_ONLY      38.6x real time  {'CPU': 799}
  64 frames, CPU_AND_GPU   51.6x real time  {'CPU': 91, 'GPU': 708}
  64 frames, CPU_AND_NE    57.9x real time  {'NeuralEngine': 778, 'CPU': 21}
 128 frames, CPU_ONLY      39.1x real time  {'CPU': 799}
 128 frames, CPU_AND_GPU   56.4x real time  {'GPU': 799}
 128 frames, CPU_AND_NE     6.2x real time  {'CPU': 798}


Model is not in eval mode. Consider calling '.eval()' on your model prior to conversion
Running MIL backend_mlprogram pipeline: 100%|##########| 12/12 [00:00<00:00, 133.55 passes/s]
Model is not in eval mode. Consider calling '.eval()' on your model prior to conversion
Running MIL backend_mlprogram pipeline: 100%|##########| 12/12 [00:00<00:00, 131.03 passes/s]


Up to 64 frames (1.6 s) nearly every operation runs on the Neural Engine, and it's the fastest of the three. At 128
the Neural Engine refuses and everything falls back to the CPU. So using it means decoding in short windows, with
overlaps cross-faded.

But the decoder normalises each channel over the whole time axis (instance norm, inside every AdaIN layer), so a
window sees its own statistics rather than the sentence's. That changes the sound:

In [ ]:
SAMPLES, HAR = 600, 120  # Audio samples and source columns per frame


@torch.no_grad()
def windowed(asr, F0, N, s, W=64, overlap=16):
    "Decode in windows of W frames, overlapping and cross-faded. The source is computed for the whole sentence."
    T = asr.shape[-1]
    har = source(F0)
    if T <= W:
        return body(asr, F0, N, s, har).squeeze()
    starts = list(range(0, T - W, W - overlap)) + [T - W]
    out, weight, ramp = torch.zeros(T * SAMPLES), torch.zeros(T * SAMPLES), torch.linspace(0, 1, overlap * SAMPLES)
    for k, a in enumerate(starts):
        y = body(
            asr[..., a : a + W],
            F0[:, 2 * a : 2 * (a + W)],
            N[:, 2 * a : 2 * (a + W)],
            s,
            har[..., a * HAR : (a + W) * HAR + 1],
        ).squeeze()
        w = torch.ones(W * SAMPLES)
        if k > 0:
            w[: len(ramp)] = ramp
        if k < len(starts) - 1:
            w[-len(ramp) :] = ramp.flip(0)
        out[a * SAMPLES : (a + W) * SAMPLES] += y * w
        weight[a * SAMPLES : (a + W) * SAMPLES] += w
    return out / weight


evaluate("PyTorch CPU, 64-frame windows", windowed)

PyTorch CPU, 64-frame windows | 7.19 | 8.29 | 4.43


A small but real drop, and the overlap length makes no difference (I tried 8 to 24 frames), so it's the statistics,
not the seams. 32-frame windows were worse again, about 3.95. For Kokoro, then, the GPU is the better choice: full
quality and nearly as fast. For a model of our own, it's a design rule: normalise without looking at the whole
sentence, and the Neural Engine is free to use.

## Smaller weights

The body is 107 MB at 16 bits. Two ways to shrink it: linear 8-bit quantisation, and palettisation, where each weight
becomes an index into a small table of values that k-means picks per layer, or per group of 16 channels.

Palettised models crash Core ML's Python bindings on the GPU with this version (a GIL error inside `predict`), so
they're measured on the CPU, where the 16-bit body is the comparison.

In [ ]:
def size_mb(m):
    path = os.path.join(tempfile.mkdtemp(), "m.mlpackage")
    m.save(path)
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs) / 1e6


def palette(nbits, **kw):
    return lambda m: cto.palettize_weights(
        m, cto.OptimizationConfig(global_config=cto.OpPalettizerConfig(nbits=nbits, mode="kmeans", **kw))
    )


linear8 = lambda m: cto.linear_quantize_weights(
    m, cto.OptimizationConfig(global_config=cto.OpLinearQuantizerConfig(mode="linear_symmetric"))
)
print(f"16-bit body: {size_mb(bodies[256]):.0f} MB")
for name, compress, units in [
    ("8-bit linear", linear8, "CPU_AND_GPU"),
    ("6-bit palette", palette(6), "CPU_ONLY"),
    ("4-bit palette, per 16 channels", palette(4, granularity="per_grouped_channel", group_size=16), "CPU_ONLY"),
]:
    small = {T: compress(m) for T, m in bodies.items()}
    print(f"{name}: {size_mb(small[256]):.0f} MB")
    evaluate(f"Core ML {name}, {units}", coreml(small, units))

16-bit body: 107 MB
8-bit linear: 54 MB
Core ML 8-bit linear, CPU_AND_GPU | 17.50 | 6.75 | 4.52
6-bit palette: 40 MB
Core ML 6-bit palette, CPU_ONLY | 28.10 | 8.01 | 4.47
4-bit palette, per 16 channels: 29 MB
Core ML 4-bit palette, per 16 channels, CPU_ONLY | 26.92 | 8.50 | 4.44


Running compression pass linear_quantize_weights: 100%|##########| 362/362 [00:00<00:00, 553.42 ops/s]
Running MIL frontend_milinternal pipeline: 0 passes [00:00, ? passes/s]
Running compression pass linear_quantize_weights: 100%|##########| 362/362 [00:00<00:00, 557.97 ops/s]
Running MIL frontend_milinternal pipeline: 0 passes [00:00, ? passes/s]
Running compression pass linear_quantize_weights: 100%|##########| 362/362 [00:00<00:00, 561.58 ops/s]
Running MIL frontend_milinternal pipeline: 0 passes [00:00, ? passes/s]
Running compression pass palettize_weights: 100%|##########| 362/362 [00:19<00:00, 18.95 ops/s]
Running MIL frontend_milinternal pipeline: 0 passes [00:00, ? passes/s]
Running compression pass palettize_weights: 100%|##########| 362/362 [00:18<00:00, 19.80 ops/s]
Running MIL frontend_milinternal pipeline: 0 passes [00:00, ? passes/s]
Running compression pass palettize_weights: 100%|##########| 362/362 [00:18<00:00, 19.64 ops/s]
Running MIL frontend_milinternal pipeline: 

8 bits costs nothing audible, and 4 bits per group of channels costs a little. On the GPU, though, 8-bit was slower
than 16-bit: Core ML expands the weights as it goes. So the trade is download size against speed, and on a phone the
16-bit model's 107 MB may matter more than its speed.

## Android: ExecuTorch and XNNPACK

Android has no Core ML. PyTorch's own on-device runtime, ExecuTorch, hands the parts of a model it can to a backend:
XNNPACK for ARM CPUs everywhere, with Vulkan and Qualcomm's NPU as other options. XNNPACK runs on this Mac too, so we
can try it here.

It needs constant weights, which is why `exportable_decoder` folds weight normalisation: without that, it accelerated
5 of the 75 convolutions. And it can't take the instance norm (`native_batch_norm` with no running statistics), nor
mean over the time axis, so those run on ExecuTorch's slow portable kernels, with a layout copy at every hand-off.

In [ ]:
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.devtools.backend_debug import get_delegation_info
from executorch.exir import to_edge_transform_and_lower
from executorch.runtime import Runtime

T = 128
asr, F0, N, s = inputs[0]
args = tuple(a.contiguous() for a in (asr[..., :T], F0[:, : 2 * T], N[:, : 2 * T], s, source(F0[:, : 2 * T])))
with torch.no_grad():
    edge = to_edge_transform_and_lower(torch.export.export(body, args), partitioner=[XnnpackPartitioner()])
print(get_delegation_info(edge.exported_program().graph_module).get_summary())
path = os.path.join(tempfile.mkdtemp(), "body.pte")
with open(path, "wb") as f:
    f.write(edge.to_executorch().buffer)
method = Runtime.get().load_program(path).load_method("forward")
method.execute(list(args))
t = time.perf_counter()
method.execute(list(args))
print(f"{os.path.getsize(path) / 1e6:.0f} MB, {T * 0.025 / (time.perf_counter() - t):.1f}x real time on 6 cores")

Total delegated subgraphs: 25
Number of delegated nodes: 845
Number of non-delegated nodes: 736

213 MB, 6.3x real time on 6 cores


[cpuinfo_utils.cpp:71] Reading file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:87] Failed to open midr file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:167] Number of efficient cores 6


That's slower than plain PyTorch on the CPU. Most of the convolutions run on XNNPACK, but over six hundred operations
around the instance norms don't. Rewriting the norm as plain means didn't help, since XNNPACK only takes means over
image-shaped tensors. The real Android numbers need a phone, and probably a different backend: Vulkan for the GPU, or
the NPU through Qualcomm's.

## Summary

In [ ]:
pd.DataFrame(results).round(2)

,decoder,x real time,vs seed floor,MOS
0,"PyTorch CPU, 1 thread",5.57,0.99,4.52
1,"PyTorch CPU, 6 threads",9.45,1.00,4.52
2,"PyTorch CPU, exportable",10.04,2.35,4.52
3,"PyTorch GPU, exportable",25.84,2.36,4.52
4,"Core ML 16-bit, CPU_ONLY",25.99,7.88,4.48
5,"Core ML 16-bit, CPU_AND_GPU",36.70,6.70,4.52
6,"PyTorch CPU, 64-frame windows",7.19,8.29,4.43
7,"Core ML 8-bit linear, CPU_AND_GPU",17.50,6.75,4.52
8,"Core ML 6-bit palette, CPU_ONLY",28.10,8.01,4.47
9,"Core ML 4-bit palette, per 16 channels, CPU_ONLY",26.92,8.50,4.44


- **The decoder is the whole job**: 90% of the time. The front half can stay on the CPU.
- **The GPU through Core ML is the winner on Apple hardware**: about four times faster than PyTorch on the CPU, at
  full quality, once the complex-number STFT is replaced and the pitch source kept in 32-bit.
- **The Neural Engine is faster still per window, but only for windows of 1.6 s or less**, and windows cost quality
  because of instance norm.
- **Weights can shrink from 107 MB to about 29 MB** with 4-bit palettisation, for a small drop in MOS.
- **Android is the open question.** The model exports to ExecuTorch, but XNNPACK leaves the instance norms on slow
  kernels.

Two lessons for the voice we train ourselves: avoid instance norm over the whole sentence, so it can run in windows
on the Neural Engine, and keep anything that accumulates over the sentence (like phase) out of the network.

## Next

- Run it on a phone: the front half and the two decoder models in an iOS app, measuring speed and battery on an older
  iPhone, not just this Mac.
- Android: try ExecuTorch's Vulkan backend, or ONNX Runtime, on a real device.
- Feed the GPU decoder into notebook 102's player.